In [1]:
import kagglehub
import re
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset

c:\Users\laran\Home\Documents\code\venv\NLPvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Download latest version
path = kagglehub.dataset_download("austinreese/goodreads-books")

print("Path to dataset files:", path)

print(os.listdir(path))

Path to dataset files: C:\Users\laran\.cache\kagglehub\datasets\austinreese\goodreads-books\versions\1
['goodreads_books.csv']


In [3]:
csv_path = os.path.join(path, "goodreads_books.csv")
df = pd.read_csv(csv_path)

#df.head()

In [4]:
df.columns

Index(['id', 'title', 'link', 'series', 'cover_link', 'author', 'author_link',
       'rating_count', 'review_count', 'average_rating', 'five_star_ratings',
       'four_star_ratings', 'three_star_ratings', 'two_star_ratings',
       'one_star_ratings', 'number_of_pages', 'date_published', 'publisher',
       'original_title', 'genre_and_votes', 'isbn', 'isbn13', 'asin',
       'settings', 'characters', 'awards', 'amazon_redirect_link',
       'worldcat_redirect_link', 'recommended_books', 'books_in_series',
       'description'],
      dtype='object')

## Clean genres

In [19]:
# Safely build input text: title + description
df["title"] = df["title"].fillna("")
df["description"] = df["description"].fillna("")
df["text"] = (df["title"] + " " + df["description"]).str.strip()

# ---------- GENRE PARSING ----------
def extract_genres(raw):
    """
    Extracts genre names from the `genre_and_votes` column.

    This is intentionally robust, but you might want to tweak it
    once you inspect a few raw values of genre_and_votes.
    """
    if pd.isna(raw):
        return []
    if not isinstance(raw, str):
        raw = str(raw)

    # Split by comma: "Fantasy (123), Young Adult (456)"
    parts = [p.strip() for p in raw.split(",") if p.strip()]

    genres = []
    for p in parts:
        # Take text up to the first "(" or up to digits
        # e.g. "Fantasy (1234)" -> "Fantasy"
        m = re.match(r"([A-Za-z0-9 &\-/]+)", p)
        if m:
            g = m.group(1).strip()
            if g:
                genres.append(g)

    # Deduplicate
    return list(dict.fromkeys(genres))

df["genres_list"] = df["genre_and_votes"].apply(extract_genres)

# Keep only rows with non-empty text and at least one genre
df = df[(df["text"].str.len() > 0) & (df["genres_list"].str.len() > 0)].reset_index(drop=True)

print("Num samples after filtering:", len(df))

Num samples after filtering: 49359


In [20]:
CANONICAL_GENRES = [
    "Fantasy",
    "Science Fiction",
    "Romance",
    "Mystery",
    "Thriller",
    "Horror",
    "Historical",
    "Contemporary",
    "Classics",
    "Young Adult",
    "Children",
    "Nonfiction",
    "Biography",
    "History",
    "Self-Help",
    "Poetry",
    "Comics/Graphic Novels",
    "LGBTQ+",
]


In [21]:
def map_raw_genre_to_canonical(raw: str) -> list[str]:
    """
    Map a single raw Goodreads genre string to one or more
    canonical genres using simple keyword rules.
    """
    g = raw.lower()
    mapped = []

    # --- main content genres ---
    if "fantasy" in g:
        mapped.append("Fantasy")

    if "science fiction" in g or "sci-fi" in g or "sci fi" in g or "scifi" in g:
        mapped.append("Science Fiction")

    if "romance" in g or "chick lit" in g:
        mapped.append("Romance")

    if "mystery" in g or "detective" in g or "crime" in g:
        mapped.append("Mystery")

    if "thriller" in g or "suspense" in g:
        mapped.append("Thriller")

    if "horror" in g:
        mapped.append("Horror")

    if "historical" in g:
        mapped.append("Historical")

    if "contemporary" in g or "realistic" in g:
        mapped.append("Contemporary")

    if "classic" in g:
        mapped.append("Classics")

    # --- age categories / audience ---
    if "young adult" in g or "ya " in g or g.startswith("ya"):
        mapped.append("Young Adult")

    if "children" in g or "kids" in g or "middle grade" in g:
        mapped.append("Children")

    # --- nonfiction-ish ---
    if "nonfiction" in g or "non-fiction" in g or "true story" in g:
        mapped.append("Nonfiction")

    if "biography" in g or "memoir" in g:
        mapped.append("Biography")

    if "history" in g:
        mapped.append("History")

    if "self help" in g or "self-help" in g or "personal development" in g:
        mapped.append("Self-Help")

    if "poetry" in g:
        mapped.append("Poetry")

    if "graphic novels" in g or "graphic novel" in g or "manga" in g or "comics" in g:
        mapped.append("Comics/Graphic Novels")

    if "lgbt" in g or "queer" in g or "glbt" in g or "gay" in g or "lesbian" in g:
        mapped.append("LGBTQ+")

    # You can add more rules as you see weird genres in the data.

    # Deduplicate but keep order
    if mapped:
        mapped = list(dict.fromkeys(mapped))
    return mapped


def map_genre_list(raw_list: list[str]) -> list[str]:
    """
    Map a list of raw genres to canonical genres.
    """
    canonical = []
    for raw in raw_list:
        canonical.extend(map_raw_genre_to_canonical(raw))
    # Deduplicate
    canonical = list(dict.fromkeys(canonical))
    return canonical


In [22]:
df["canonical_genres"] = df["genres_list"].apply(map_genre_list)


In [23]:
df = df[df["canonical_genres"].str.len() > 0].reset_index(drop=True)
print("Num samples after canonical mapping:", len(df))

from collections import Counter
genre_counts = Counter(g for gens in df["canonical_genres"] for g in gens)
print("Canonical genres and counts:", genre_counts)


Num samples after canonical mapping: 45768
Canonical genres and counts: Counter({'Fantasy': 13413, 'Romance': 13024, 'Young Adult': 9717, 'Nonfiction': 8185, 'Contemporary': 7295, 'Historical': 6825, 'Mystery': 6052, 'Science Fiction': 5253, 'Classics': 4856, 'Children': 4307, 'Thriller': 3968, 'History': 3268, 'Biography': 3008, 'Horror': 2230, 'Comics/Graphic Novels': 1980, 'Poetry': 1349, 'Self-Help': 1010, 'LGBTQ+': 977})


# Transformer settings and training

In [24]:
import wandb
wandb.init(
    project="genre-classifier",
    settings=wandb.Settings(code_dir=".", _disable_stats=True),
    config={},   # avoids logging big Trainer config
)

In [25]:
# ---------- CONFIG ----------
DATA_PATH = "goodreads_books.csv"   # path to your Kaggle CSV
OUTPUT_DIR = "genre_model"
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
TEST_SIZE = 0.1
RANDOM_SEED = 42

# ---------- MULTI-LABEL BINARIZATION ----------
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["canonical_genres"])

# Save label binarizer so we can decode predictions later
joblib.dump(mlb, os.path.join(OUTPUT_DIR, "label_binarizer.pkl"))
print("Saved MultiLabelBinarizer with", len(mlb.classes_), "classes.")
print("Classes:", mlb.classes_)

Saved MultiLabelBinarizer with 18 classes.
Classes: ['Biography' 'Children' 'Classics' 'Comics/Graphic Novels' 'Contemporary'
 'Fantasy' 'Historical' 'History' 'Horror' 'LGBTQ+' 'Mystery' 'Nonfiction'
 'Poetry' 'Romance' 'Science Fiction' 'Self-Help' 'Thriller' 'Young Adult']


In [ ]:
# ---------- TRAIN/TEST SPLIT ----------
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED
)

# ---------- TOKENIZER ----------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(texts):
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

# ---------- DATASETS ----------
train_encodings = tokenize_batch(train_texts)
val_encodings = tokenize_batch(val_texts)

class BookDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx]).float()
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = BookDataset(train_encodings, train_labels)
val_dataset = BookDataset(val_encodings, val_labels)

# ---------- MODEL ----------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(mlb.classes_),
    problem_type="multi_label_classification"
)

# ---------- METRICS (optional but useful) ----------
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)

    # Explicitly cast to float (not big arrays)
    return {
        "micro_f1": float(micro_f1),
        "macro_f1": float(macro_f1),
    }

# ---------- TRAINING ----------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    logging_steps=100,
    report_to=["wandb"],          # keep this if you want wandb
    logging_first_step=True,
    log_on_each_node=False,
    # prediction_loss_only=True,  <-- REMOVE THIS
    save_total_limit=1,           # optional: keep only best/last checkpoint
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\laran\Home\Documents\code\venv\NLPvenv\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer.train()

# ---------- SAVE MODEL + TOKENIZER ----------
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Training complete. Model saved to", OUTPUT_DIR)

Epoch,Training Loss,Validation Loss


In [ ]:
logs = pd.DataFrame(trainer.state.log_history)
logs.tail()

In [ ]:
import matplotlib.pyplot as plt

train_logs = logs[logs["loss"].notna()]
eval_logs = logs[logs["eval_loss"].notna()]

plt.figure()
plt.plot(train_logs["step"], train_logs["loss"], label="train_loss")
plt.plot(eval_logs["step"], eval_logs["eval_loss"], label="val_loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
f1_logs = logs[logs["micro_f1"].notna()]

print(f1_logs[["epoch", "micro_f1", "macro_f1"]])


In [ ]:
import os
import torch
import joblib
import numpy as np
from typing import List, Tuple
from transformers import AutoTokenizer, AutoModelForSequenceClassification


class GenreClassifier:
    def __init__(self, model_dir: str = "genre_model", device: str | None = None):
        self.model_dir = model_dir

        # Device handling
        if device is not None:
            self.device = torch.device(device)
        else:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load artifacts
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        self.model.to(self.device)
        self.model.eval()

        self.mlb = joblib.load(os.path.join(model_dir, "label_binarizer.pkl"))
        self.label_names = list(self.mlb.classes_)

    @torch.no_grad()
    def predict(
        self,
        title: str | None,
        description: str | None,
        threshold: float = 0.5,
        top_k: int | None = None,
    ) -> List[Tuple[str, float]]:
        """
        Returns a list of (genre, probability) for genres above threshold.
        If top_k is set, returns at most top_k genres sorted by prob desc.
        """
        title = title or ""
        description = description or ""
        text = (title + " " + description).strip()
        if not text:
            return []

        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=256,
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        logits = self.model(**inputs).logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]

        # Collect genres above threshold
        indices = np.where(probs >= threshold)[0].tolist()

        candidates = [(self.label_names[i], float(probs[i])) for i in indices]

        # If nothing passes threshold, pick the single best genre
        if not candidates:
            best_idx = int(np.argmax(probs))
            candidates = [(self.label_names[best_idx], float(probs[best_idx]))]

        # Sort and truncate
        candidates.sort(key=lambda x: x[1], reverse=True)
        if top_k is not None:
            candidates = candidates[:top_k]

        return candidates

    @torch.no_grad()
    def predict_labels_only(
        self,
        title: str | None,
        description: str | None,
        threshold: float = 0.5,
        top_k: int | None = None,
    ) -> List[str]:
        """Convenience: just returns the genre names."""
        return [g for g, _ in self.predict(title, description, threshold, top_k)]


In [ ]:
if __name__ == "__main__":
    clf = GenreClassifier("genre_model")
    genres = clf.predict("The Name of the Wind", "A high fantasy novel about Kvothe...")
    print(genres)
